# Yelp 레스토랑 리뷰 전처리

Yelp Open Dataset에서 레스토랑 리뷰를 추출·정제하여 프롬프트 증강용 시드 데이터셋(`sampled_df.parquet`)을 만든다.

**파이프라인**: 레스토랑 필터 → 메타데이터 병합 → 기간 필터(2014~2016) → state=PA 선택 → 텍스트 정제 → 레스토랑별 최대 100개 cap → 리뷰 길이 20~300단어 → 평점별 2,000개 균형 샘플링 → 셔플 후 저장.

**산출물**: `yelp_dataset/sampled_df.parquet` (10,000행 = 평점 1~5 × 2,000개)

### 라이브러리 불러오기

In [ ]:
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"

from pathlib import Path

import emoji
import pandas as pd
import matplotlib.pyplot as plt

### 데이터 경로 설정

In [ ]:
DATA_DIR = Path("../data")

# 원본 데이터 경로
BUSINESS_PATH = DATA_DIR / "yelp_academic_dataset_business.json"
REVIEW_PATH = DATA_DIR / "yelp_academic_dataset_review.json"

# 산출물 경로
OUTPUT_PATH = DATA_DIR / "sampled_df.parquet"

### Yelp business / review 데이터 로드

In [ ]:
business_df = pd.read_json(BUSINESS_PATH, lines=True)
print("Business shape:", business_df.shape)
business_df.head()

In [ ]:
review_df = pd.read_json(REVIEW_PATH, lines=True)
print("Review shape:", review_df.shape)
review_df.head()

### 레스토랑 카테고리만 필터링

- `categories`에 "Restaurants"가 포함된 비즈니스만 선택

In [ ]:
restaurant_df = business_df[
    business_df["categories"].notna()
    & business_df["categories"].str.contains("Restaurants", case=False, na=False)
].copy()

print("Restaurant business shape:", restaurant_df.shape)

### 비즈니스 메타데이터 컬럼명 정리

- 리뷰 평점과 식당 평균 평점이 모두 `stars`이므로 병합 전에 비즈니스 컬럼명을 구분

In [ ]:
restaurant_df = restaurant_df.rename(
    columns={
        "name": "business_name",
        "stars": "business_stars",
        "review_count": "business_review_count",
    }
)

### 레스토랑 리뷰만 필터링 후 비즈니스 메타데이터와 병합

In [ ]:
restaurant_review_df = review_df[
    review_df["business_id"].isin(restaurant_df["business_id"])
].copy()

print("Restaurant review shape:", restaurant_review_df.shape)
print("Number of reviewed restaurants:", restaurant_review_df["business_id"].nunique())

merged_df = restaurant_review_df.merge(restaurant_df, on="business_id", how="inner")
print("Merged shape:", merged_df.shape)

### 리뷰 컬럼명 정리

In [ ]:
merged_df = merged_df.rename(
    columns={"stars": "review_stars", "text": "review_text", "date": "review_date"}
)

merged_df[["review_id", "business_id", "business_name", "review_stars", "review_text", "review_date", "state"]].head()

### 기간 필터링 (2014-01-01 ~ 2016-12-31)

In [ ]:
merged_df["review_date"] = pd.to_datetime(merged_df["review_date"])

START_DATE = pd.Timestamp("2014-01-01")
END_DATE = pd.Timestamp("2016-12-31 23:59:59")

period_df = merged_df[
    merged_df["review_date"].between(START_DATE, END_DATE)
].copy()

print("Period filtered shape:", period_df.shape)

### 분석 대상 state 선택 (PA)

- 후보 state 비교 결과, 평점별 균형 샘플 확보에 유리한 **PA**를 최종 선택

In [ ]:
df = period_df[period_df["state"] == "PA"].copy()

print("PA shape:", df.shape)
print(df["review_stars"].value_counts().sort_index())

### 기본 텍스트 정제

- 이모지 제거, 공백 정규화, 빈 리뷰 및 비정상 평점 제거

In [ ]:
def clean_text(df):
    df = df.copy()
    df["review_text"] = df["review_text"].astype(str).str.strip()
    df["review_text"] = df["review_text"].apply(lambda s: emoji.replace_emoji(s, replace=""))
    df["review_text"] = df["review_text"].str.replace(r"\s+", " ", regex=True).str.strip()
    df = df[df["review_text"].notna()]
    df = df[df["review_text"] != ""]
    df = df[df["review_stars"].isin([1, 2, 3, 4, 5])]
    df["review_stars"] = df["review_stars"].astype(int)
    return df


work_df = clean_text(df)
print("After basic cleaning:", work_df.shape)
print(work_df["review_stars"].value_counts().sort_index())

### 단어 수 계산

In [ ]:
def count_words(text):
    return len(str(text).split())


work_df["word_count"] = work_df["review_text"].apply(count_words)
print(work_df["word_count"].describe())

### 레스토랑별 최대 리뷰 수 제한 (100개)

- 특정 인기 식당 리뷰가 과대표집되지 않도록 식당별 최대 100개로 cap

In [ ]:
MAX_REVIEWS_PER_BUSINESS = 100
RANDOM_STATE = 42

sampled_indices = []
for business_id, group in work_df.groupby("business_id", sort=False):
    n_sample = min(len(group), MAX_REVIEWS_PER_BUSINESS)
    sampled_indices.extend(group.sample(n=n_sample, random_state=RANDOM_STATE).index)

work_df = work_df.loc[sampled_indices].reset_index(drop=True)

print("After business cap:", work_df.shape)
print("Number of businesses:", work_df["business_id"].nunique())
print(work_df["review_stars"].value_counts().sort_index())

### 리뷰 길이 필터링 (20 ~ 300 단어)

In [ ]:
work_df["word_count"] = work_df["review_text"].apply(count_words)

MIN_WORDS = 20
MAX_WORDS = 300

work_df = work_df[
    (work_df["word_count"] >= MIN_WORDS) & (work_df["word_count"] <= MAX_WORDS)
].copy()

print("After length filtering:", work_df.shape)
print(work_df["word_count"].describe())
print(work_df["review_stars"].value_counts().sort_index())

### 단어 수 분포 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(work_df["word_count"], bins=60, color="steelblue", edgecolor="white")
axes[0].axvline(work_df["word_count"].median(), color="red", linestyle="--", label=f'median={work_df["word_count"].median():.0f}')
axes[0].axvline(work_df["word_count"].mean(), color="orange", linestyle="--", label=f'mean={work_df["word_count"].mean():.1f}')
axes[0].set_title("Word Count Distribution")
axes[0].set_xlabel("word_count"); axes[0].set_ylabel("frequency"); axes[0].legend()

upper = work_df["word_count"].quantile(0.99)
stars_sorted = sorted(work_df["review_stars"].unique())
data_by_star = [work_df.loc[work_df["review_stars"] == s, "word_count"] for s in stars_sorted]
axes[1].boxplot(data_by_star, labels=stars_sorted, showfliers=False)
axes[1].set_ylim(0, upper)
axes[1].set_title("Word Count by Review Stars")
axes[1].set_xlabel("review_stars"); axes[1].set_ylabel("word_count")

plt.tight_layout(); plt.show()

### 평점별 균형 샘플링 (평점당 2,000개)

In [ ]:
N_PER_RATING = 2000
RANDOM_STATE = 42

rating_counts = work_df["review_stars"].value_counts().sort_index()
actual_n_per_rating = min(N_PER_RATING, rating_counts.min())
print("Available per rating:\n", rating_counts)
print("Actual per rating:", actual_n_per_rating)

sampled_indices = []
for rating, group in work_df.groupby("review_stars", sort=False):
    sampled_indices.extend(group.sample(n=actual_n_per_rating, random_state=RANDOM_STATE).index)

sampled_df = work_df.loc[sampled_indices].copy()
sampled_df = sampled_df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print("Sampled shape:", sampled_df.shape)
print(sampled_df["review_stars"].value_counts().sort_index())

### 저장

In [ ]:
sampled_df.to_parquet(OUTPUT_PATH, index=False)
print("Saved:", OUTPUT_PATH)
print("Shape:", sampled_df.shape)
print("State:", sampled_df["state"].unique())
print("word_count range:", sampled_df["word_count"].min(), "~", sampled_df["word_count"].max())